# Axial v3 Iteration B - low-cost experiments

Framework reproducible para B0-B6 usando solo train/validation.

In [ ]:
from pathlib import Path
import json

from lumbar_mri.axial_v3.architectures import ArchitectureConfig, build_axial_v3_model
from lumbar_mri.axial_v3.guards import require_train_val_only
from lumbar_mri.axial_v3.losses import combined_segmentation_loss, tversky_loss
from lumbar_mri.axial_v3.low_cost import SelectionGuardrail, apply_raw0_effective_weight, cap_class_weight_ratio, passes_other_class_guardrail, validation_ranking_key
from lumbar_mri.axial_v3.presence import Raw0PresenceHead, total_loss_with_presence
from lumbar_mri.axial_v3.registry import REGISTRY_COLUMNS, registry_row_from_config

ALLOWED_SPLITS = require_train_val_only(["train", "val"], context="notebook_52")
CONFIG_PATH = Path("config") / "axial_v3_low_cost_grid.json"
REGISTRY_PATH = Path("outputs") / "axial_v3" / "experiment_registry.csv"

In [ ]:
def load_low_cost_grid(path=CONFIG_PATH):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    require_train_val_only(payload["developmentSplits"], context="notebook_52.config")
    return payload


def build_baseline_model(in_channels=1):
    return build_axial_v3_model(ArchitectureConfig(name="axial_unet2d", in_channels=in_channels))


def ranking_key(validation_metrics):
    return validation_ranking_key(validation_metrics)


def prepare_class_weights(base_weights, raw0_multiplier=1.0, max_ratio=None):
    weighted = apply_raw0_effective_weight(base_weights, raw0_index=1, multiplier=raw0_multiplier)
    return cap_class_weight_ratio(weighted, max_ratio=max_ratio)

## Selection policy

Selection is validation-only: maximize foreground Dice, then reduce raw_0 predicted-in-absent cases, then improve raw_0 precision, then preserve Dice excluding raw_0. Guardrail values are loaded from JSON and are not hidden in notebook code.